# 8.1 Fehlende Daten

Realistische Datensätze sind oft unvollständig. In einer Umfrage hat eine Person
mit einer Frage nichts anfangen können und daher nichts angekreuzt. Oder ein
Messsensor an der Produktionsanlage ist abends ausgefallen, was erst am nächsten
Morgen bemerkt wird. Es gibt viele Gründe, warum Datensätze unvollständig sind.
In diesem Abschnitt beschäftigen wir uns damit, fehlende Daten aufzuspüren und
lernen einfache Methoden kennen, damit umzugehen.

## Fehlende Daten aufspüren mit isnull()

Wir arbeiten im Folgenden mit einem echten Datensatz der Verkaufsplattform
[Autoscout24.de](https://www.autoscout24.de), der Verkaufsdaten zu 1000 Autos
enthält.

In [ ]:
import pandas as pd

daten = pd.read_csv('autoscout24_fehlende_daten.csv')
daten.info()

Der Datensatz enthält 1000 Autos, also 1000 Zeilen bzw. 1000 Datenpunkte, und 14
Merkmale. Wir hatten bereits festgestellt, dass die Anzahl der
`non-null`-Einträge für die verschiedenen Merkmale unterschiedlich ist.
Offensichtlich ist nur bei 963 Autos eine »Farbe« eingetragen und die »Leistung
(PS)« ist nur bei 987 Autos gültig. Am wenigsten gültige Einträge hat das
Merkmal »Verbrauch (l/100 km)«, wohingegen bei der Eigenschaft »Kilometerstand
(km)« nur ein ungültiger Eintrag auftaucht. Welche Einträge ungültig sind,
können wir mit der Methode `isnull()` bestimmen. Die Methode liefert ein
Pandas-DataFrame zurück, das True/False-Werte enthält. True steht dabei dafür,
dass ein Wert fehlt bzw. mit dem Eintrag `NaN` gekennzeichnet ist (= not a
number). Weitere Details finden Sie in der [Pandas-Dokumentation →
isnull()](https://pandas.pydata.org/docs/reference/api/pandas.isnull.html).

In [ ]:
# TODO: ???  (fehlende Einträge im Datensatz markieren)

Bereits in der zweiten Zeile befindet sich ein Auto, bei dem das Merkmal
»Verbrauch (l/100 km)« nicht gültig ist (ggf. müssen Sie weiter nach rechts
scrollen), denn dort steht `True`. Wir betrachten uns diesen Eintrag:

In [ ]:
daten.loc[1]

Bei dem Auto handelt es sich um ein Hybrid-Fahrzeug, vielleicht wurde deshalb
der »Verbrauch (l/100 km)« nicht angegeben? Ist das vielleicht auch bei den
anderen Autos der Grund? Wir speichern zunächst die `isnull()`-Datenstruktur in
einer eigenen Variable ab und ermitteln, wie viele Autos keinen gültigen Eintrag
bei diesem Merkmal haben. Dazu nutzen wir aus, dass der boolesche Wert `False`
bei Rechnungen als 0 interpretiert wird und der boolesche Wert `True` als 1. Die
Methode `.sum()` summiert pro Spalte alle Werte, so dass sie direkt die Anzahl
der ungültigen Werte pro Spalte liefert.

In [ ]:
fehlende_daten = daten.isnull()

# TODO: ???  (Anzahl der fehlenden Werte pro Spalte bestimmen)

Jetzt lassen wir uns diese 109 Autos anzeigen, bei denen ungültige Werte beim
»Verbrauch (l/100 km)« angegeben wurden. Dazu nutzen wir die True-Werte in der
Spalte `Verbrauch (l/100 km)` als Filter für den ursprünglichen Datensatz und
speichern den gefilterten Datensatz in der Variable
`autos_mit_fehlendem_verbrauch_pro_100km`. Zumindest die ersten 20 Autos lassen
wir uns dann mit der `.head(20)`-Methode anzeigen.

In [ ]:
autos_mit_fehlendem_verbrauch_pro_100km = daten[fehlende_daten['Verbrauch (l/100 km)']]
autos_mit_fehlendem_verbrauch_pro_100km.head(20)

Beim Kraftstoff werden alle möglichen Angaben gemacht: Hybrid, Benzin, Diesel
und Elektro. Wir müssten jetzt systematisch den fehlenden Angaben nachgehen. Für
Elektrofahrzeuge und ggf. Hybridautos ist die Angabe »Verbrauch (l/100 km)«
unsinnig. Aber das zweite Auto mit der Nr. 5 wird mit Benzin betrieben, da
scheint Nachlässigkeit beim Ausfüllen der Merkmale vorzuliegen. Beim fünften
Auto mit der Nr. 77 ist zwar der »Verbrauch (l/100 km)« nicht angegeben, aber
dafür der »Verbrauch (g/km)«. Daraus könnten wir den »Verbrauch (l/100 km)«
abschätzen und den fehlenden Wert ergänzen. Es gibt verschiedene Strategien, mit
fehlenden Daten umzugehen. Die beiden wichtigsten Verfahren zum Umgang mit
fehlenden Daten sind

1. Löschen (Elimination) und
2. Vervollständigung (Imputation).

Bei Elimination werden Datenpunkte (Autos) und/oder Merkmale gelöscht. Bei
Imputation (Vervollständigung) werden die fehlenden Werte ergänzt. Beide
Verfahren werden wir nun etwas detaillierter betrachten.

## Löschen (Elimination) mit drop()

Bei der Elimination (Löschen) können wir filigran vorgehen oder die
Holzhammer-Methode verwenden. Beispielsweise können wir entscheiden, das Merkmal
»Verbrauch (l/100 km)« komplett zu löschen und einfach nur den »Verbrauch
(g/km)« zu berücksichtigen. Aber ein kurzer Blick auf die Daten hatte ja bereits
gezeigt, dass diese Werte auch nur unzuverlässig gefüllt waren, auch wenn sie
technisch gültig sind. Wir löschen beide Merkmale. Dazu benutzen wir die Methode
`drop()` mit dem zusätzlichen Argument `columns=['Verbrauch (l/100 km)',
'Verbrauch (g/km)']`. Da wir gleich zwei Spalten auf einmal eliminieren möchten,
müssen wir die Spalten (Columns) als Liste übergeben. Danach überprüfen wir mit
der Methode `.info()`, ob das Löschen geklappt hat.

In [ ]:
# TODO: ???  (Spalten Verbrauch (l/100 km) und Verbrauch (g/km) löschen)
daten.info()

Leider hat der Befehl `drop()` nicht funktioniert! Was ist da los? Python
verfolgt das Programmierparadigma »Explizit ist besser als implizit!« Daher
werden zwar durch den `drop()`-Befehl die beiden Spalten gelöscht, aber der
Datensatz `daten` selbst bleibt aus Sicherheitsgründen unverändert. Möchten wir
den Datensatz mit den gelöschten Merkmalen weiter verwenden, müssen wir ihn in
einer neuen Variable speichern oder die alte Variable `daten` damit
überschreiben. Wir nehmen eine neue Variable namens `daten_ohne_verbrauch`.

In [ ]:
daten_ohne_verbrauch = daten.drop(columns=['Verbrauch (l/100 km)', 'Verbrauch (g/km)'])
daten_ohne_verbrauch.info()

Ein weiterer Datenpunkt weist einen ungültigen Eintrag für den »Kilometerstand
(km)« auf. Schauen wir zunächst nach, um welches Auto es sich handelt.

In [ ]:
daten_ohne_verbrauch[ daten_ohne_verbrauch['Kilometerstand (km)'].isnull() ]

Bei den Einträgen des Autos sind noch mehr Probleme ersichtlich. Die
Erstzulassung war sicherlich nicht bei 37.500 km und das Jahr ist nicht 12/2020.
Wir können jetzt diesen Datenpunkt löschen oder den Datenpunkt reparieren. Das
Reparieren einzelner Werte behandeln wir hier nicht, wir entscheiden uns für das
Löschen. Standardmäßig löscht die `drop()`-Methode ohnehin Zeilen, also
Datenpunkte, so dass wir ohne weitere Optionen den Index der zu löschenden
Datenpunkte angeben. Diesmal verwenden wir die alte Variable um den reduzierten
Datensatz zu speichern.

In [ ]:
daten_ohne_verbrauch = # TODO: ???  (Auto mit dem Index 708 löschen)
daten_ohne_verbrauch.info()

Weitere Optionen der `drop()`-Methode finden Sie in der [Pandas-Dokumentation →
drop()](https://pandas.pydata.org/docs/reference/api/pandas.DataFrame.drop.html).
In diesem Skript bleiben wir durchgängig bei der Neuzuweisung, speichern das
Ergebnis von `drop()` also immer in einer Variable.

Ob alle Angaben plausibel sind, ist nicht gesagt. Bei dem Peugeot mit dem Index
708 hatten wir ja gesehen, dass bei der Erstzulassung eine Kilometerangabe
stand. Tatsächlich gab es bereits erste Hinweise darauf, dass manche Werte
technisch gültig, aber nicht plausibel sind. Die Spalte mit dem Jahr
beispielsweise wurde beim Import als Datentyp Object klassifiziert. Zu erwarten
wäre jedoch der Datentyp Integer gewesen. Schauen wir noch einmal in den
ursprünglichen Datensatz hinein.

In [ ]:
daten['Jahr'].unique()

Da bei dem Peugeot mit dem Index 708 das Jahr fälschlicherweise mit `12/2020`
angegeben wurde, hat dieser Texteintrag dazu geführt, dass die komplette
Spalte als Object klassifiziert wurde und nicht als Integer. Selbst nachdem wir
den Peugeot gelöscht haben, bleibt die Spalte »Jahr« vom Datentyp Object, bis
wir sie gezielt in einen Zahlentyp umwandeln. Daher müssen stets weitere
Plausibilitätsprüfungen durchgeführt werden, bevor die Daten genutzt werden, um
statistische Aussagen zu treffen oder ein ML-Modell zu trainieren.

## Vervollständigung (Imputation) mit fillna()

Auch bei den Angaben zur Farbe fehlen Einträge. Zum Beispiel die Zeile mit
dem Index 2 ist unvollständig.

In [ ]:
daten_ohne_verbrauch.loc[2]

Diesmal entscheiden wir uns dazu, diese Eigenschaft nicht wegzulassen.
ML-Verfahren brauchen immer einen gültigen Wert und nicht `NaN`. Wir müssen
daher den fehlenden Wert ersetzen. Eine Möglichkeit ist, eine Farbe zu erfinden,
zum Beispiel 'bunt', oder die fehlenden Werte explizit durch einen Eintrag 'keine
Angabe' zu vervollständigen. Dazu benutzen wir die Methode `fillna()` (siehe
[Pandas-Dokumentation →
fillna()](https://pandas.pydata.org/docs/reference/api/pandas.DataFrame.fillna.html)).
Die Vervollständigung soll nur die NaN-Werte der Spalte »Farbe« füllen. Daher
filtern wir zuerst diese Spalte und wenden darauf die `fillna()`-Methode an. Das
erste Argument der `fillna()`-Methode ist der Wert, durch den die NaN-Werte
ersetzt werden sollen (hier `'keine Angabe'`). Damit die Vervollständigung
explizit gespeichert wird, überschreiben wir die Spalte.

In [ ]:
daten_ohne_verbrauch['Farbe'] = # TODO: ???  (fehlende Farben durch 'keine Angabe' ersetzen)

# Kontrolle der Vervollständigung
daten_ohne_verbrauch.isnull().sum()

Wenn wir uns jetzt noch einmal die dritte Zeile ansehen, sehen wir, dass
`fillna()` funktioniert hat.

In [ ]:
daten_ohne_verbrauch.loc[2,:]

Bei den PS-Zahlen haben wir ebenfalls keine vollständigen Daten. Diesmal haben
wir nicht kategoriale Daten wie die Farben, sondern numerische Werte. Daher
bietet es sich hier eine zweite Methode der Ersetzung (Imputation) an. Wenn wir
überall da, wo keine PS-Zahlen vorliegen, den Mittelwert der vorhandenen
PS-Zahlen einsetzen, verändern wir zumindest den Mittelwert des gesamten
Datensatzes nur wenig. Wir berechnen daher zuerst den Mittelwert mit der Methode
`.mean()` und nutzen dann die `fillna()`-Methode.

In [ ]:
mittelwert = daten_ohne_verbrauch['Leistung (PS)'].mean()
print(f'Der Mittelwert der vorhandenen Einträge »Leistung (PS)« ist: {mittelwert:.2f}')

daten_ohne_verbrauch['Leistung (PS)'] = daten_ohne_verbrauch['Leistung (PS)'].fillna(mittelwert)

Noch einmal die Kontrolle, ob jetzt alle NaN-Werte eliminiert oder
vervollständigt wurden:

In [ ]:
daten_ohne_verbrauch.isnull().sum()

Der Mittelwert der »Leistung (PS)« liegt deutlich über dem Median. Möglicherweise
haben wir den Datensatz durch die Ersetzung mit dem Mittelwert eher
verschlechtert: Wir haben die Streuung der Werte verkleinert und vermutlich auch
Zusammenhänge zwischen den Merkmalen verzerrt. Es gibt bessere Alternativen:

* Statt des Mittelwerts den **Median** einsetzen, der weniger empfindlich
  gegenüber Ausreißern ist.
* Die Autos mit fehlenden PS-Zahlen zunächst weglassen, für die übrigen Autos ein
  **lineares Regressionsmodell oder einen Entscheidungsbaum** trainieren und damit
  die fehlenden PS-Zahlen abschätzen.
* Die fehlenden Werte aus der Spalte »Leistung (kW)« **umrechnen**. Das ist hier
  die beste Lösung, denn »Leistung (PS)« und »Leistung (kW)« sind dasselbe
  Merkmal in zwei Einheiten. Genau genommen ist eine der beiden Spalten
  überflüssig und wir könnten »Leistung (PS)« auch ganz entfernen.

## Zusammenfassung und Ausblick

Ein wichtiger Teil eines ML-Projektes beschäftigt sich mit der Aufbereitung der
Daten für die ML-Algorithmen. Dabei ist es nicht nur wichtig, in großen
Datensammlungen fehlende Einträge aufspüren zu können, sondern ein Gespür dafür
zu entwickeln, wie mit den fehlenden Daten umgegangen werden soll. Die
Strategien hängen dabei von der Anzahl der fehlenden Daten und ihrer Bedeutung
ab. Häufig werden unvollständige Daten aus der Datensammlung gelöscht
(Elimination) oder numerische Einträge durch den Mittelwert der vorhandenen
Daten ersetzt (Imputation). Wie kategoriale Daten für ML-Algorithmen aufbereitet
werden müssen, wird im nächsten Kapitel erklärt.

## Mini-Übungen

### Mini-Übung 1

Übertragen Sie das Vorgehen auf einen Datensatz aus der Fertigung. Die Datei
`3ddruck_fehlende_daten.csv` enthält 200 3D-Druckaufträge mit jeweils 15
Merkmalen. Auch hier fehlen Einträge.

1. Lesen Sie die Datei ein. Die Spalte `Nummer` soll als Zeilenindex dienen.
2. Verschaffen Sie sich einen Überblick über den Datensatz. Bei welchen
   Merkmalen fehlen Werte?
3. Ermitteln Sie, wie viele Werte pro Merkmal fehlen.
4. Lassen Sie sich alle Druckaufträge anzeigen, bei denen die Angabe zur
   Druckzeit fehlt.

In [ ]:
# Code-Zelle

### Mini-Übung 2

Arbeiten Sie mit dem Datensatz aus der vorherigen Mini-Übung weiter.

1. Das Merkmal `Oberflaechenguete` hat viele Lücken und lässt sich ohnehin aus
   der Schichthöhe ableiten. Entfernen Sie dieses Merkmal und arbeiten Sie ab
   jetzt mit einer bereinigten Kopie des Datensatzes weiter.
2. Genau ein Merkmal hat nur einen einzigen fehlenden Wert. Finden Sie den
   zugehörigen Druckauftrag.
3. Sehen Sie sich diesen Datenpunkt genau an. Welche zwei Einträge sind
   auffällig?
4. Sehen Sie sich die Werte der Spalte `Drucktemperatur (C)` an. Warum wurde
   diese Spalte beim Import nicht als Zahl, sondern als Text (`object`)
   eingelesen?
5. Entfernen Sie den auffälligen Druckauftrag aus dem bereinigten Datensatz.

In [ ]:
# Code-Zelle

### Mini-Übung 3

Arbeiten Sie mit dem bereinigten Datensatz aus der vorherigen Mini-Übung weiter.

1. Die fehlenden Angaben zur `Farbe` sollen erhalten bleiben. Vervollständigen
   Sie sie mit einem passenden Ersatzwert.
2. Bei der `Druckzeit (min)` handelt es sich um numerische Werte, deren
   Verteilung stark rechtsschief ist, es gibt also einige wenige sehr lange
   Drucke. Vervollständigen Sie die fehlenden Werte mit einer geeigneten
   Kennzahl der vorhandenen Werte und begründen Sie Ihre Wahl.
3. Überprüfen Sie, dass der Datensatz jetzt keine fehlenden Werte mehr enthält.

In [ ]:
# Code-Zelle